# Capítulo 5 — ALTER: Reescrevendo o passado

**Livro:** começa na página **195** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")
from banco import preparar
motor = preparar(5, "gabarito", recomecar=True)  # preparar(..., recomecar=True) volta o capítulo ao início
%reload_ext sql
%config SqlMagic.displaylimit = 60
%config SqlMagic.feedback = 0
%sql motor

## Alterações de tabelas · PDF p. 200

#### Exercício: ADD COLUMN

Acrescente a coluna `telefone VARCHAR(15)` em `meus_contatos` e mostre a estrutura.

In [ ]:
%%sql
ALTER TABLE meus_contatos ADD COLUMN telefone VARCHAR(15);
SELECT column_name AS coluna, data_type AS tipo, character_maximum_length AS tamanho,
       is_nullable AS aceita_nulo, column_default AS padrao
FROM information_schema.columns
WHERE table_schema = current_schema() AND table_name = 'meus_contatos'
ORDER BY ordinal_position;

## Reconstrução total de tabelas · PDF p. 201

#### Exercício: Reformando o calhambeque

A tabela `calhambeques` está mal feita. Reforme: renomeie para `carros`; acrescente `carro_id SERIAL PRIMARY KEY`; renomeie `mod` para `modelo`; mude `ano` para `INT` e `preco` para `DECIMAL(9,2)` (texto vira número com `USING ano::int` e `USING preco::numeric`).

In [ ]:
%%sql
ALTER TABLE calhambeques RENAME TO carros;
ALTER TABLE carros ADD COLUMN carro_id SERIAL PRIMARY KEY;
ALTER TABLE carros RENAME COLUMN mod TO modelo;
ALTER TABLE carros
    ALTER COLUMN ano TYPE INT USING ano::int,
    ALTER COLUMN preco TYPE DECIMAL(9,2) USING preco::numeric;
SELECT * FROM carros;

## Renomeando a tabela · PDF p. 201

#### Exercício: RENAME TO

A tabela `projetos` passa a se chamar `lista_projetos`.

In [ ]:
%%sql
ALTER TABLE projetos RENAME TO lista_projetos;
SELECT * FROM lista_projetos;

## Reinstrumentalizando nossas colunas · PDF p. 203

#### Exercício: Renomear coluna e criar chave

Renomeie `numero` para `proj_id` e transforme-a em chave primária (`ADD PRIMARY KEY (proj_id)`).

In [ ]:
%%sql
ALTER TABLE lista_projetos RENAME COLUMN numero TO proj_id;
ALTER TABLE lista_projetos ADD PRIMARY KEY (proj_id);
SELECT column_name AS coluna, data_type AS tipo, character_maximum_length AS tamanho,
       is_nullable AS aceita_nulo, column_default AS padrao
FROM information_schema.columns
WHERE table_schema = current_schema() AND table_name = 'lista_projetos'
ORDER BY ordinal_position;

## ALTER e CHANGE · PDF p. 204

#### Exercício: O CHANGE do MySQL em dois comandos

O `CHANGE` do MySQL renomeia e muda o tipo de uma vez. Aqui: renomeie `descricaodoprojeto` para `descricao` e mude o tipo para `VARCHAR(100)` (`ALTER COLUMN ... TYPE`). Faça o mesmo com `empreiteiranaobra` → `empreiteira`.

In [ ]:
%%sql
ALTER TABLE lista_projetos RENAME COLUMN descricaodoprojeto TO descricao;
ALTER TABLE lista_projetos ALTER COLUMN descricao TYPE VARCHAR(100);
ALTER TABLE lista_projetos RENAME COLUMN empreiteiranaobra TO empreiteira;
SELECT column_name AS coluna, data_type AS tipo, character_maximum_length AS tamanho,
       is_nullable AS aceita_nulo, column_default AS padrao
FROM information_schema.columns
WHERE table_schema = current_schema() AND table_name = 'lista_projetos'
ORDER BY ordinal_position;

## Altere duas colunas com apenas um comando SQL · PDF p. 205

#### Exercício: Várias mudanças num ALTER

Num **único** `ALTER TABLE`, acrescente `data_inicio DATE` e `orcamento DECIMAL(9,2)` e mude `empreiteira` para `VARCHAR(60)` (separe as ações por vírgula).

In [ ]:
%%sql
ALTER TABLE lista_projetos
    ADD COLUMN data_inicio DATE,
    ADD COLUMN orcamento DECIMAL(9,2),
    ALTER COLUMN empreiteira TYPE VARCHAR(60);
SELECT column_name AS coluna, data_type AS tipo, character_maximum_length AS tamanho,
       is_nullable AS aceita_nulo, column_default AS padrao
FROM information_schema.columns
WHERE table_schema = current_schema() AND table_name = 'lista_projetos'
ORDER BY ordinal_position;

## Rápido! Largue esta coluna! · PDF p. 208

#### Exercício: DROP COLUMN

Remova a coluna `data_inicio`.

In [ ]:
%%sql
ALTER TABLE lista_projetos DROP COLUMN data_inicio;
SELECT column_name AS coluna, data_type AS tipo, character_maximum_length AS tamanho,
       is_nullable AS aceita_nulo, column_default AS padrao
FROM information_schema.columns
WHERE table_schema = current_schema() AND table_name = 'lista_projetos'
ORDER BY ordinal_position;

## Algumas funções de texto muito úteis · PDF p. 214

#### Exercício: Funções de texto

Aplique a `'  Recife, PE  '`: `TRIM`, `LENGTH` (antes e depois do TRIM), `UPPER`, `LOWER`, `REVERSE`, `LEFT(...,6)`, `RIGHT(...,2)` (depois do TRIM), `SUBSTRING(... FROM 1 FOR 3)` e `split_part(..., ',', 1)`.

In [ ]:
%%sql
SELECT TRIM('  Recife, PE  ') AS aparado,
       LENGTH('  Recife, PE  ') AS tam_antes, LENGTH(TRIM('  Recife, PE  ')) AS tam_depois,
       UPPER('Recife, PE') AS maiusc, LOWER('Recife, PE') AS minusc, REVERSE('Recife, PE') AS invertido,
       LEFT('Recife, PE', 6) AS esquerda, RIGHT(TRIM('  Recife, PE  '), 2) AS direita,
       SUBSTRING('Recife, PE' FROM 1 FOR 3) AS pedaco, split_part('Recife, PE', ',', 1) AS antes_da_virgula;

## Utilize uma coluna atual para preencher uma nova coluna · PDF p. 217

#### Exercício: UPDATE com funções

Acrescente `cidade VARCHAR(40)` e `uf CHAR(2)` em `meus_contatos` e preencha a partir de `local` com `UPDATE ... SET uf = RIGHT(local, 2), cidade = split_part(local, ',', 1)`.

In [ ]:
%%sql
ALTER TABLE meus_contatos ADD COLUMN cidade VARCHAR(40), ADD COLUMN uf CHAR(2);
UPDATE meus_contatos SET uf = RIGHT(local, 2), cidade = split_part(local, ',', 1);
SELECT primeiro_nome, local, cidade, uf FROM meus_contatos;

## Como funciona nosso combo UPDATE e SET? · PDF p. 218

#### Exercício: Removendo a coluna não atômica

Agora que `cidade` e `uf` existem, apague `local`. Conte quantos contatos há por `uf`.

In [ ]:
%%sql
ALTER TABLE meus_contatos DROP COLUMN local;
SELECT uf, COUNT(*) AS contatos FROM meus_contatos GROUP BY uf ORDER BY contatos DESC;